In [2]:
import time, numpy as np, pandas as pd, torch, librosa
from speechbrain.utils.fetching import LocalStrategy
from pathlib import Path
from speechbrain.inference.speaker import EncoderClassifier

ROOT = Path("..")
model = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir=str(ROOT / "models/ecapa"),
    local_strategy=LocalStrategy.COPY
)

c:\Projets\VoicePrint_Piscine\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mateo\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [3]:
def empreinte(chemin):
    y, sr = librosa.load(ROOT / chemin, sr=16000)   # même chargement que ta baseline
    signal = torch.tensor(y).unsqueeze(0)            # forme attendue : (1, nb_échantillons)
    with torch.no_grad():                            # pas d'entraînement, juste un calcul
        return model.encode_batch(signal).squeeze()  # 192 nombres

def cosinus(a, b):
    return torch.nn.functional.cosine_similarity(a, b, dim=0).item()

In [4]:
meta = pd.read_csv(ROOT / "data/metadata.csv")
train = meta[meta["split"] == "train"]
locs = train["speaker_id"].unique()[:6]

memes, diff = [], []
for i in range(5):
    A = train[train["speaker_id"] == locs[i]]["wav_path"].tolist()
    B = train[train["speaker_id"] == locs[i + 1]]["wav_path"].tolist()
    ea1, ea2, eb = empreinte(A[0]), empreinte(A[1]), empreinte(B[0])
    memes.append(cosinus(ea1, ea2))
    diff.append(cosinus(ea1, eb))

print("Même personne  :", np.round(memes, 3), "→ moyenne", round(np.mean(memes), 3))
print("Personnes diff.:", np.round(diff, 3), "→ moyenne", round(np.mean(diff), 3))

Même personne  : [0.38  0.618 0.49  0.621 0.68 ] → moyenne 0.558
Personnes diff.: [ 0.244  0.311  0.108 -0.07   0.045] → moyenne 0.127


In [5]:
y, sr = librosa.load(ROOT / A[0], sr=16000, duration=3.0)
t = time.perf_counter()
with torch.no_grad():
    model.encode_batch(torch.tensor(y).unsqueeze(0))
print("Temps pour 3 s d'audio :", round(time.perf_counter() - t, 3), "s")

Temps pour 3 s d'audio : 0.291 s
